> **Current notebook role — 2026-11-12.** This notebook retains the upstream stage code used by the manuscript implementation. Selected figure production uses `run.py` in `code_release/iCCA-molecular-subtyping` or the public source repository. See `docs/NOTEBOOK_STATUS.md` in that code repository for stage ownership. Earlier execution outputs are preserved in the archived notebook snapshot.

# Step 4 — Genomic, proteomic and matched multi-omic characterization

## tl;dr

This notebook tests whether the locked transcriptomic states have supporting DNA, mass-spectrometry protein, targeted RPPA and matched multi-omic structure. Each assay retains its own eligible denominator. Principal C1/C2/C3 states define headline contrasts; C1/C2, C1/C3 and C2/C3 intermediate states are shown only as continuity evidence.

The rebuilt analysis corrects four publication-blocking problems in the historical notebook: mutation-record-only denominators, mutation counts mislabeled as TMB, unmatched/incompatible protein matrices, and random multi-omic feature selection. MOFA now exports its model, factor scores, weights, view-wise variance and convergence trace. Unsupervised multi-omic clusters are reported as sensitivity evidence and do not replace locked transcriptomic labels.

## Context & Methods

### Analysis ownership and guardrails

- Step 1 expression and subtype assignments are locked read-only inputs.
- DNA event frequencies include every assay-profiled sample, including samples with no recorded nonsynonymous event.
- Variant totals are called *nonsynonymous variant counts*, not TMB, because harmonized callable territory is unavailable.
- Mass-spectrometry proteins are filtered by study coverage, standardized within source study, and fitted with Study in the subtype contrast model.
- TCGA-CHOL RPPA is a separate targeted-platform validation layer, not pooled with mass spectrometry.
- MOFA is restricted to the 2022Dong RNA/protein/CNV triple-matched cohort and uses deterministic high-variance features.
- Biological interpretation remains tied to governed C1/C2/C3 labels, not arbitrary algorithm cluster numbers.

In [ ]:
rm(list = ls())
options(stringsAsFactors = FALSE)
project_root <- normalizePath(getwd())
step_dir <- file.path(project_root, "4-Multiomic")
table_dir <- file.path(step_dir, "publication", "tables")
figure_dir <- file.path(step_dir, "publication", "figures")
model_dir <- file.path(step_dir, "publication", "models")
refresh_step4 <- identical(Sys.getenv("SCLUCID_STEP4_REFRESH", "0"), "1")
required <- c("IRdisplay")
missing <- required[!vapply(required, requireNamespace, logical(1), quietly = TRUE)]
if (length(missing)) stop("Missing R packages: ", paste(missing, collapse = ", "))
cat("Execution mode:", ifelse(refresh_step4, "full Step4 refresh", "validation/read-only"), "
")

In [ ]:
if (refresh_step4) {
  core_status <- system2("Rscript", file.path(step_dir, "scripts", "run_step4_core_analysis.R"))
  if (core_status != 0) stop("Step4 core analysis failed with status ", core_status)

  python_candidates <- unique(c(
    Sys.getenv("SCLUCID_MOFA_PYTHON", unset = NA_character_),
    file.path(Sys.getenv("HOME"), "micromamba", "envs", "omicverse", "bin", "python"),
    Sys.which("python3")
  ))
  python_candidates <- python_candidates[!is.na(python_candidates) & file.exists(python_candidates)]
  supports_mofa <- vapply(python_candidates, function(exe) {
    identical(system2(exe, c("-c", shQuote("import mofapy2")), stdout = FALSE, stderr = FALSE), 0L)
  }, logical(1))
  if (!any(supports_mofa)) stop("No Python interpreter with mofapy2. Set SCLUCID_MOFA_PYTHON.")
  mofa_python <- python_candidates[which(supports_mofa)[1]]
  mofa_status <- system2(mofa_python, file.path(step_dir, "scripts", "run_step4_mofa.py"))
  if (mofa_status != 0) stop("Step4 MOFA analysis failed with status ", mofa_status)

  final_status <- system2("Rscript", file.path(step_dir, "scripts", "finalize_step4_analysis.R"))
  if (final_status != 0) stop("Step4 finalization failed with status ", final_status)
}

In [ ]:
required_outputs <- file.path(table_dir, c(
  "step4_input_contract.csv", "step4_assay_specific_inclusion.csv",
  "step4_driver_snv_indel_study_stratified_tests.csv",
  "step4_mass_spec_study_adjusted_deg_main_subtypes.csv",
  "step4_matched_rna_protein_subtype_concordance.csv",
  "step4_tcga_rppa_subtype_associations.csv",
  "step4_mofa_factor_scores.csv", "step4_mofa_feature_weights.csv",
  "step4_mofa_variance_explained_per_factor.csv",
  "step4_multiomic_cluster_comparison.csv",
  "step4_multiomic_release_gate_status.csv"
))
stopifnot(all(file.exists(required_outputs)), all(file.info(required_outputs)$size > 0))

## Data

### 1. Locked interfaces and assay-specific denominators

In [ ]:
input_contract <- read.csv(file.path(table_dir, "step4_input_contract.csv"), check.names = FALSE)
assay_inclusion <- read.csv(file.path(table_dir, "step4_assay_specific_inclusion.csv"), check.names = FALSE)
stopifnot(all(input_contract$ReadOnly))
input_contract
assay_inclusion

The denominators are intentionally not pooled: DNA SNV/indel uses 359 profiled tumors across three studies; mass spectrometry uses 292 matched tumors across two studies; TCGA RPPA uses 25 matched tumors; and MOFA uses 204 triple-matched 2022Dong tumors. Principal-state sample sizes are recorded beside each total.

### 2. Historical-code audit and repaired interfaces

In [ ]:
legacy_audit <- read.csv(file.path(table_dir, "step4_legacy_static_audit_findings.csv"), check.names = FALSE)
protein_scale <- read.csv(file.path(table_dir, "step4_protein_scale_audit.csv"), check.names = FALSE)
legacy_audit
protein_scale

The protected Step0 protein object is preserved for provenance. Because the two mass-spectrometry sources use incompatible numeric scales, publication-facing effect estimates use the Step4-derived within-study standardized matrix. This repair does not overwrite Step0 QC artifacts.

## Results

### 3. Driver SNV/indel frequencies and per-sample variant counts

In [ ]:
driver_tests <- read.csv(file.path(table_dir, "step4_driver_snv_indel_study_stratified_tests.csv"), check.names = FALSE)
variant_counts <- read.csv(file.path(table_dir, "step4_nonsynonymous_variant_count_by_sample.csv"), check.names = FALSE)
data.frame(
  AssayProfiledN = nrow(variant_counts),
  ZeroRecordedNonsynonymousEvents = sum(variant_counts$NonsynonymousVariantCount == 0),
  FDRSignificantDriverContrasts = sum(driver_tests$CMHFDR < 0.05, na.rm = TRUE)
)
head(driver_tests[order(driver_tests$CMHFDR, driver_tests$CMHPValue), ], 12)

In [ ]:
IRdisplay::display_png(file = file.path(figure_dir, 'step4_driver_snv_indel_frequency_main_subtypes.png'))

In [ ]:
IRdisplay::display_png(file = file.path(figure_dir, 'step4_nonsynonymous_variant_counts_by_subtype_and_study.png'))

These panels support descriptive genomic heterogeneity. In the full assay roster, 11 of 45 prespecified subtype-versus-other driver contrasts meet within-subtype FDR < 0.05 (C2: TP53, KRAS, IDH1, IDH2, BAP1, SMAD4; C3: TP53, KRAS, IDH1, BAP1, SMAD4; C1: none). These are study-stratified, panel-limited associations rather than causal or genome-wide driver claims. The historical record-bearing analysis used smaller, mutation-positive-only denominators; restored mutation-negative samples and study adjustment explain the changed P/FDR values (see step4_legacy_vs_rebuilt_mutation_test_comparison.csv). Cross-study mutation counts remain sensitive to panel design and sequencing depth, hence the explicit Study display and the prohibition on a TMB label.

#### TMB feasibility audit

In [ ]:
tmb_readiness <- read.csv(file.path(table_dir, 'step4_tmb_calculation_readiness.csv'), check.names = FALSE)
tmb_readiness
stopifnot(tmb_readiness$Available[tmb_readiness$Requirement == 'Per-sample nonsynonymous variant count'] == 'YES',
          tmb_readiness$Available[tmb_readiness$Requirement == 'Source-provided Deng-only total_perMB field'] == 'YES (Deng only)',
          all(tmb_readiness$Available[!tmb_readiness$Requirement %in% c('Per-sample nonsynonymous variant count', 'Source-provided Deng-only total_perMB field')] == 'NO'))

The current inputs support per-sample nonsynonymous variant counts across the harmonized roster and contain a source-provided `total_perMB`/`log2_TMB` field for 2023Deng only (139 mutation records, 79 WES-assay samples). The Deng-only metric cannot be recalculated or pooled with Dong/TCGA-CHOL until its callable-territory and filtering provenance are verified. A publication-grade cross-study TMB is therefore not available; the historical `captureSize = 50` value is not used.

### 4. 2022Dong CNV burden and driver-level gains/losses

In [ ]:
cnv_burden <- read.csv(file.path(table_dir, "step4_cnv_burden_2022dong.csv"), check.names = FALSE)
driver_cnv <- read.csv(file.path(table_dir, "step4_driver_cnv_frequency_main_subtypes_2022dong.csv"), check.names = FALSE)
head(driver_cnv[order(-pmax(driver_cnv$GainFraction, driver_cnv$LossFraction)), ], 15)

In [ ]:
IRdisplay::display_png(file = file.path(figure_dir, 'step4_driver_cnv_frequency_2022dong.png'))

CNV inference is cohort-specific because the current continuous GISTIC2 source is available for 2022Dong. It is supporting multi-omic evidence rather than a cross-study genomic validation claim.

### 5. Study-adjusted mass-spectrometry subtype features

In [ ]:
protein_deg <- read.csv(file.path(table_dir, "step4_mass_spec_study_adjusted_deg_main_subtypes.csv"), check.names = FALSE)
protein_counts <- as.data.frame(table(
  Subtype = protein_deg$Subtype[protein_deg$FDR < 0.05],
  Direction = protein_deg$Direction[protein_deg$FDR < 0.05]
))
protein_counts <- protein_counts[protein_counts$Freq > 0, ]
protein_counts
head(protein_deg[order(protein_deg$FDR, -abs(protein_deg$StandardizedDifference)), ], 15)

In [ ]:
IRdisplay::display_png(file = file.path(figure_dir, 'step4_mass_spec_deg_main_subtypes.png'))

In [ ]:
IRdisplay::display_png(file = file.path(figure_dir, 'step4_mass_spec_marker_heatmap.png'))

### 6. RNA–protein concordance

In [ ]:
rna_protein <- read.csv(file.path(table_dir, "step4_matched_rna_protein_subtype_concordance.csv"), check.names = FALSE)
pathway_concordance <- read.csv(file.path(table_dir, "step4_matched_rna_protein_pathway_concordance_summary.csv"), check.names = FALSE)
rna_protein
pathway_concordance

In [ ]:
IRdisplay::display_png(file = file.path(figure_dir, 'step4_matched_rna_protein_contrast_concordance.png'))

In [ ]:
IRdisplay::display_png(file = file.path(figure_dir, 'step4_matched_rna_protein_hallmark_concordance.png'))

Matched subtype contrasts show positive cross-layer concordance for all three principal states. This supports preservation of subtype programs at the protein level without implying that every transcriptomic marker has a concordant protein effect.

### 7. TCGA-CHOL targeted RPPA validation

In [ ]:
rppa_samples <- read.csv(file.path(table_dir, "step4_tcga_rppa_matched_samples.csv"), check.names = FALSE)
rppa_tests <- read.csv(file.path(table_dir, "step4_tcga_rppa_subtype_associations.csv"), check.names = FALSE)
data.frame(TotalMatched = nrow(rppa_samples),
           PrincipalStates = sum(rppa_samples$SubtypeClass == "Main"),
           FDRSignificantTargetContrasts = sum(rppa_tests$FDR < 0.05, na.rm = TRUE))
head(rppa_tests[order(rppa_tests$FDR, rppa_tests$PValue), ], 12)

In [ ]:
IRdisplay::display_png(file = file.path(figure_dir, 'step4_tcga_rppa_pca.png'))

TCGA RPPA is now summarized separately. Its small matched denominator makes it a targeted supportive check, not a replacement for mass-spectrometry evidence or a high-powered discovery cohort.

### 8. Matched MOFA latent factors

In [ ]:
mofa_variance <- read.csv(file.path(table_dir, "step4_mofa_total_variance_explained.csv"), check.names = FALSE)
mofa_associations <- read.csv(file.path(table_dir, "step4_mofa_factor_subtype_associations.csv"), check.names = FALSE)
mofa_variance
mofa_associations

In [ ]:
IRdisplay::display_png(file = file.path(figure_dir, 'step4_mofa_variance_explained.png'))

In [ ]:
IRdisplay::display_png(file = file.path(figure_dir, 'step4_mofa_factor_scatter.png'))

In [ ]:
IRdisplay::display_png(file = file.path(figure_dir, 'step4_mofa_factor_state_heatmap.png'))

The filled C1/C2/C3 points define the principal-state comparison. Open intermediate-state points test whether the governed transitional states occupy plausible latent-factor continua; they do not contribute to the principal subtype association tests.

### 9. MOFA feature weights and unsupervised sensitivity

In [ ]:
IRdisplay::display_png(file = file.path(figure_dir, 'step4_mofa_top_feature_weights.png'))

In [ ]:
factor_enrichment <- read.csv(file.path(table_dir, "step4_mofa_factor_hallmark_enrichment.csv"), check.names = FALSE)
head(factor_enrichment[order(factor_enrichment$FDR), ], 15)

In [ ]:
IRdisplay::display_png(file = file.path(figure_dir, 'step4_mofa_factor_hallmark_enrichment.png'))

In [ ]:
cluster_comparison <- read.csv(file.path(table_dir, "step4_multiomic_cluster_comparison.csv"), check.names = FALSE)
cluster_comparison

In [ ]:
IRdisplay::display_png(file = file.path(figure_dir, 'step4_mofa_cluster_agreement.png'))

MOFA weights identify features contributing to latent variation, not differential-test significance. Adjusted Rand indices quantify partial agreement between unsupervised multi-omic partitions and the locked transcriptomic labels. Low or moderate agreement is biologically informative and is not treated as a failed reproduction of Step 1.

## Takeaways

- Step 4 now provides denominator-correct genomic summaries, repaired cross-study proteomics, separate TCGA RPPA evidence and a reproducible matched MOFA analysis.
- RNA–protein subtype contrasts are positively concordant across C1, C2 and C3, supporting cross-layer program preservation.
- Genomic and RPPA results remain assay- and cohort-limited; nominal signals are not promoted beyond their multiple-testing evidence.
- Multi-omic latent factors provide complementary structure. Their algorithmic clusters are sensitivity analyses, not replacement subtype labels.
- All intermediate-state panels are supplementary continuity evidence; principal-state inference remains restricted to C1/C2/C3.

## Release gate

In [ ]:
gate <- read.csv(file.path(table_dir, "step4_multiomic_release_gate_status.csv"), check.names = FALSE)
gate
stopifnot(!any(gate$Status == "BLOCK"))
manifest <- read.csv(file.path(table_dir, "step4_publication_output_manifest.csv"), check.names = FALSE)
stopifnot(all(manifest$Bytes > 0), !anyDuplicated(manifest$Artifact))
cat("Step 4 release gate: 0 blocking failures;", nrow(manifest), "tracked publication artifacts.
")
sessionInfo()